# 03 — Cross-country robustness

Identical specifications estimated separately for each country, before any
pooled model is considered.

The order is deliberate. A pooled coefficient imposes homogeneous transmission
dynamics; where countries genuinely differ, it is a weighted average of different
processes rather than a common parameter. Looking at the spread first makes that
visible instead of hiding it in one number.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from wage_transmission.config import load_models_config
from wage_transmission.cross_country import (
    estimate_country_robustness,
    estimate_panel_fixed_effects,
    summarise_country_robustness,
)

PROCESSED = Path("../data/processed/panel.csv")
ILLUSTRATIVE = not PROCESSED.exists()

if not ILLUSTRATIVE:
    panel = pd.read_csv(PROCESSED)
else:
    # No processed multi-country panel in this checkout. The frame below exists
    # only to exercise the interface: it is SIMULATED, and nothing estimated
    # from it is evidence about any real country.
    rng = np.random.default_rng(20260824)
    years = np.arange(1995, 2025)
    frames = []
    for code, beta in {"AAA": 0.4, "BBB": 0.7, "CCC": 0.9, "DDD": 1.1}.items():
        growth = rng.normal(0.017, 0.017, len(years))
        wage_growth = 0.001 + beta * growth + rng.normal(0, 0.007, len(years))
        frames.append(
            pd.DataFrame(
                {
                    "country": code,
                    "year": years,
                    "real_wage": 20000 * np.exp(np.cumsum(wage_growth)),
                    "productivity": 30 * np.exp(np.cumsum(growth)),
                }
            )
        )
    panel = pd.concat(frames, ignore_index=True)

config = load_models_config(Path("../config/models.yml"))
print("SIMULATED DATA — NOT EVIDENCE" if ILLUSTRATIVE else f"Source: {PROCESSED}")
print(f"Countries: {sorted(panel['country'].unique())}")

SIMULATED DATA — NOT EVIDENCE
Countries: ['AAA', 'BBB', 'CCC', 'DDD']


## Country-specific estimates

In [2]:
estimates = estimate_country_robustness(panel, config=config)
columns = [
    "country",
    "nobs",
    "distributed_lag_cumulative",
    "distributed_lag_cumulative_se",
    "cointegration_5pct",
]
estimates.loc[:, columns].sort_values("distributed_lag_cumulative").round(3)

,country,nobs,distributed_lag_cumulative,distributed_lag_cumulative_se,cointegration_5pct
1,BBB,30,0.693,0.210,True
0,AAA,30,0.771,0.188,False
2,CCC,30,0.894,0.178,False
3,DDD,30,1.456,0.165,False


## How different are they?

`I²` is the share of the observed variation in country estimates that exceeds what
sampling error alone would produce. A high value means a pooled number is
describing heterogeneous processes.

In [3]:
summary = summarise_country_robustness(estimates)
print(f"Countries              : {summary.n_countries}")
print(f"Median transmission    : {summary.median_cumulative_transmission:.3f}")
print(
    f"Interquartile range    : {summary.q25_cumulative_transmission:.3f}"
    f" to {summary.q75_cumulative_transmission:.3f}"
)
print(
    f"Random-effects estimate: {summary.random_effect_estimate:.3f}"
    f" (se {summary.random_effect_std_error:.3f})"
)
print(f"I-squared              : {summary.i_squared_percent:.1f}%")
print(f"Verdict                : {summary.interpretation}")

Countries              : 4
Median transmission    : 0.832
Interquartile range    : 0.751 to 1.034
Random-effects estimate: 0.965 (se 0.181)
I-squared              : 74.1%
Verdict                : moderate_cross_country_heterogeneity


## Only now: the pooled panel estimate

Country fixed effects remove country means, so the coefficient is identified from
within-country variation only, and the standard errors are clustered by country.

Cluster-robust inference is asymptotic in the **number of clusters**. With the
country counts available here — well below the conventional threshold of about 30
— the clustered errors are downward-biased, and the result says so in its own
`interpretation` field rather than leaving the caveat to a footnote.

In [4]:
pooled = estimate_panel_fixed_effects(panel)
print(f"Pooled elasticity : {pooled.elasticity:.3f}")
print(f"Clustered se      : {pooled.std_error:.3f}")
print(f"95% interval      : {pooled.lower_95:.3f} to {pooled.upper_95:.3f}")
print(f"Clusters          : {pooled.n_countries}")
print(f"Within R-squared  : {pooled.within_r_squared:.3f}")
print(f"Caveat            : {pooled.interpretation}")

Pooled elasticity : 0.801
Clustered se      : 0.110
95% interval      : 0.585 to 1.018
Clusters          : 4
Within R-squared  : 0.737
Caveat            : pooled_estimate_few_clusters_standard_errors_optimistic


In [5]:
# Year effects absorb common annual shocks, at the cost of the cross-country
# common component a global slowdown would show up in.
with_time = estimate_panel_fixed_effects(panel, time_effects=True)
print(f"Without year effects: {pooled.elasticity:.3f}")
print(f"With year effects   : {with_time.elasticity:.3f}")

Without year effects: 0.801
With year effects   : 0.797


## What this notebook does not establish

The pooled estimate is a robustness check, never a replacement for the
country-specific table above it. If the two disagree, the country estimates are
the finding and the pooled number is the artefact of assuming homogeneity.